<a href="https://colab.research.google.com/github/eltongaspar/python/blob/Advpl/GROQ_IA_Agentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

✅ 1. Instalar dependências (Google Colab)

In [ ]:
!pip install groq gradio

✅ 2. Código completo: Chat estilo WhatsApp com 2 agentes Groq

In [ ]:
import gradio as gr
from groq import Groq

# --------- SUA API KEY ---------
GROQ_API_KEY = "GROQ_API_KEY"
# -------------------------------

client = Groq(api_key=GROQ_API_KEY)

def agent_response(model, system_message, messages):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system_message}] + messages
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"❌ ERRO NO MODELO ({model}): {str(e)}"


def chat(user_message, history):

    messages = []
    for hum, bot in history:
        messages.append({"role": "user", "content": hum})
        messages.append({"role": "assistant", "content": bot})

    messages.append({"role": "user", "content": user_message})

    agent1_reply = agent_response(
        model="llama-3.1-8b-instant",
        system_message="Você é o Agente 1, técnico e objetivo.",
        messages=messages
    )

    agent2_reply = agent_response(
        model="llama-3.3-70b-versatile",
        system_message="Você é o Agente 2, crítico e analítico.",
        messages=[
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": agent1_reply}
        ]
    )

    final_message = (
        f"🤖 *Agente 1*: {agent1_reply}\n\n"
        f"🤖 *Agente 2*: {agent2_reply}"
    )

    history.append((user_message, final_message))
    return "", history


# ✅ CSS SEM ERRO — fundo escuro, letra vermelha
css = """
/* Fundo geral da interface */
.gradio-container, body {
    background-color: #000000 !important;
}

/* Fundo do bloco do chatbot */
gr-chatbot, .chatbot, .wrap.svelte-drgfj3 {
    background-color: #000000 !important;
}

/* Mensagem do usuário */
.message.user {
    background-color: #111111 !important;
    color: #FF0000 !important;   /* 🔴 vermelho forte */
    font-weight: 700 !important;
    font-size: 17px !important;
    border-radius: 10px !important;
    padding: 12px !important;
}

/* Mensagem dos agentes */
.message.bot {
    background-color: #222222 !important;
    color: #FF0000 !important;   /* 🔴 vermelho forte */
    font-weight: 700 !important;
    font-size: 17px !important;
    border-radius: 10px !important;
    padding: 12px !important;
    border: 1px solid #444444 !important;
}
"""


with gr.Blocks(css=css) as demo:

    gr.Markdown("## 💬 Chat com 2 Agentes (Tema Preto + Texto Vermelho)")

    chatbot = gr.Chatbot(height=480)
    user_input = gr.Textbox(placeholder="Digite sua mensagem...")

    user_input.submit(chat, [user_input, chatbot], [user_input, chatbot])

demo.launch()